# 第 10 章：树形任务队列与流水线调度——动手实验

本 Notebook 依次完成环境检查、工程阅读、Python reference 模拟、910B 目标构建、数据生成和双缓冲流水线设备运行。设备命令需要在已安装 CANN 的 Linux/NPU 环境执行。

## 1. 检查实验环境

本单元定位实验根目录和源码目录，并显示 CANN 环境变量。若当前环境尚未安装 CANN，先阅读后续单元和 `910b_guide.md`，不要把个人绝对路径写入代码。

In [ ]:
import os
from pathlib import Path

def locate_lab_dir():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src' / 'tree_queue_lab').is_dir():
            return candidate
    raise FileNotFoundError('无法定位树形任务队列实验根目录')

LAB_DIR = locate_lab_dir()

SRC_DIR = LAB_DIR / 'src' / 'tree_queue_lab'
script_dir = SRC_DIR / 'scripts'
print('LAB_DIR =', LAB_DIR)
print('SRC_DIR =', SRC_DIR)
print('ASCEND_HOME_PATH =', os.environ.get('ASCEND_HOME_PATH', '<未设置>'))
assert (script_dir / 'build_ops.sh').is_file()
assert (script_dir / 'run_lab.py').is_file()
print('环境与目录检查通过')

In [ ]:
# msopgen 拒绝在含非 ASCII 字符、或 group/other 可写的路径下生成算子工程。
# 若当前实验目录（或其祖先路径）不满足要求，自动整章复制到 /tmp 下的英文
# 目录并修正权限，再继续后续构建；已满足时保持原位，不产生任何复制。
import shutil
import tempfile
from pathlib import Path as _Path

def _msopgen_safe(root):
    try:
        current = _Path(root).resolve()
    except OSError:
        return False
    while True:
        text = str(current)
        if any(ord(ch) > 127 for ch in text):
            return False
        try:
            mode = os.stat(current).st_mode
        except OSError:
            return False
        if current == current.parent:  # 已到文件系统根
            break
        if (mode & 0o022) and not (mode & 0o1000):  # 非 sticky 却 group/other 可写
            return False
        current = current.parent
    return True

if not _msopgen_safe(LAB_DIR):
    reloc = _Path(tempfile.gettempdir()) / f'cannlab_tree_queue_lab_{os.getpid()}'
    if reloc.exists():
        shutil.rmtree(reloc)
    shutil.copytree(
        LAB_DIR, reloc,
        ignore=shutil.ignore_patterns('build', 'generated', 'data',
                                      '__pycache__', '*.pyc', '.git'),
    )
    for root_dir, dirs, files in os.walk(reloc):
        os.chmod(root_dir, 0o755)
        for name in files:
            os.chmod(_Path(root_dir) / name, 0o644)
    os.chdir(reloc)
    LAB_DIR = reloc
    SRC_DIR = LAB_DIR / 'src' / 'tree_queue_lab'
    script_dir = SRC_DIR / 'scripts'
    print(f'检测到 msopgen 不接受的路径，已复制到英文临时目录：{reloc}')
    print('LAB_DIR =', LAB_DIR)
else:
    print('实验路径满足 msopgen 要求，直接原位执行。')


## 2. 查看工程结构

工程源码位于 `src/tree_queue_lab`。`scripts/` 保存 Python reference（数据生成、BFS frontier、堆调度、流水线模拟、结果校验），`custom_ops/src` 保存 910B Host/Kernel 源码，`aclnn_runner` 是 ACLNN 设备运行器。

In [ ]:
import shlex
import subprocess

listing = subprocess.run(
    ['bash', '-lc', f'ls -R {shlex.quote(str(SRC_DIR))}'],
    text=True, capture_output=True, check=True
)
print(listing.stdout)

## 3. 阅读关键源码

先看 Python reference 的调度核心 `scheduler.py`，再看 910B 的 Host tiling 和 Kernel。Host 侧计算 `taskCount` 并通过 tiling 数据传入 `queueDepth`、`computeLanes`；Kernel 用一个 control block 保持跨任务时序，不在 Kernel 中写死树规模和层数。

In [ ]:
files = [
    'scripts/scheduler.py',
    'custom_ops/src/TreeQueuePipelineLite/op_host/tree_queue_pipeline_lite.cpp',
    'custom_ops/src/TreeQueuePipelineLite/op_kernel/tree_queue_pipeline_lite.cpp',
    'aclnn_runner/main_tree_queue_benchmark.cpp',
]
for relative in files:
    path = SRC_DIR / relative
    print(f'\n===== {relative} =====')
    shown = subprocess.run(
        ['bash', '-lc', f'cat {shlex.quote(str(path))} | head -45'],
        text=True, capture_output=True, check=True
    )
    print(shown.stdout)

## 4. Python reference：BFS frontier 与优先队列调度

先用参考实现跑通完整的树依赖、BFS frontier、FIFO/优先队列顺序和双缓冲流水线模拟。`levels` 体现层级依赖，`priority_order` 可能不同于 BFS 顺序，但父节点一定排在子节点之前。

In [ ]:
import json
import subprocess
import sys

data_dir = SRC_DIR / 'data'
subprocess.run([
    sys.executable, str(script_dir / 'gen_data.py'),
    '--num_nodes', '31', '--seed', '10', '--output', str(data_dir)
], check=True)
subprocess.run([
    sys.executable, str(script_dir / 'run_lab.py'),
    '--data_dir', str(data_dir), '--output', str(data_dir / 'output.json')
], check=True)

result = json.loads((data_dir / 'output.json').read_text(encoding='utf-8'))
print('level sizes =', [len(level) for level in result['levels']])
print('BFS order =', result['bfs_order'])
print('priority order =', result['priority_order'])
print('FIFO end_to_end =', result['fifo_pipeline']['end_to_end'])
print('priority end_to_end =', result['priority_pipeline']['end_to_end'])

## 5. 为 910B 编译自定义算子

下面命令默认使用 `TARGET=ascend910b`。脚本会调用 `msopgen` 生成工程，复制 `TreeQueuePipelineLite` 的 Host/Kernel 源码并构建安装。

In [ ]:
build = subprocess.run(
    ['bash', 'scripts/build_ops.sh'],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(build.stdout[-2000:] if len(build.stdout) > 2000 else build.stdout)
print(build.stderr)
# 若失败发生在 "Install custom OPP package" 阶段，请确认安装目标是本实验的
# custom_ops/generated/local_opp（build_ops.sh 使用 --install-path，不写系统
# CANN 的 opp/vendors）；仍失败时再检查 CANN 与 msopgen 环境。
assert build.returncode == 0, '算子构建失败：请查看上方输出定位具体阶段。'


## 6. 编译 ACLNN runner

Runner 负责构造 ACL tensor、调用 `aclnnTreeQueuePipelineLite` 并复制 `stage_end` / `dependency_ok` 回 Host。构建前需要 `source scripts/env_custom_opp.sh` 指向已安装的自定义 OPP 路径。

In [ ]:
runner = subprocess.run(
    ['bash', 'scripts/build_runner.sh'],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(runner.stdout)
print(runner.stderr)
assert runner.returncode == 0, 'runner 构建失败，请检查 custom OPP 安装路径'

## 7. 生成设备输入数据

`gen_data.py` 生成 `data/input` 下的 `parent.bin`、`cost.bin`、`order_fifo.bin`、`order_priority.bin` 以及 Python reference 的 `ref_stage_end_*.bin` 供设备对比。

In [ ]:
data = subprocess.run(
    [sys.executable, 'scripts/gen_data.py', '--num_nodes', '31', '--seed', '10', '--output', 'data'],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(data.stdout)
print(data.stderr)
assert data.returncode == 0

## 8. 运行设备 benchmark：priority 与 fifo

该命令在 910B 上实际执行 `TreeQueuePipelineLite`，对比设备 `stage_end` 与 Python reference。时间为设备实际运行结果，数值会随 CANN 版本、频率和设备状态变化，但 `max_error=0` 与 `dependency value=1` 应稳定出现。

下一单元在同一 bash 进程里先 `source scripts/env_custom_opp.sh` 再运行 runner，确保 `ASCEND_CUSTOM_OPP_PATH` 指向本实验的 `local_opp`。


In [ ]:
# 必须在同一个 bash 进程里 source env_custom_opp.sh 再运行 runner：
# runner 依赖 ASCEND_CUSTOM_OPP_PATH 指向本实验 local_opp 的算子注册信息，
# 否则 ACLNN 找不到 TreeQueuePipelineLite -> 161001 / nnopExecutor == nullptr。
for mode in ('priority', 'fifo'):
    benchmark = subprocess.run(
        'source scripts/env_custom_opp.sh && '
        f'aclnn_runner/build/main_tree_queue_benchmark data {mode}',
        cwd=SRC_DIR, shell=True, executable='/bin/bash',
        text=True, capture_output=True,
    )
    print(benchmark.stdout)
    print(benchmark.stderr)
    assert benchmark.returncode == 0, f'{mode} benchmark 失败'


## 9. 结果验证

最后用校验脚本检查父子依赖、堆序和流水线阶段关系。完成后进入 `10.03_chapter_test.ipynb`。本 Notebook 只提供验证入口，不预置设备运行结果。

In [ ]:
verify = subprocess.run(
    [sys.executable, str(script_dir / 'verify_results.py'), str(data_dir / 'output.json')],
    cwd=SRC_DIR,
    text=True,
    capture_output=True,
)
print(verify.stdout)
print(verify.stderr)
assert verify.returncode == 0

## 10. 课后实践：比较队列深度

使用相同的 `priority_order` 和任务耗时，分别将 `queue_depth` 设置为 1、2、3，记录 `end_to_end`。说明队列深度增大为什么会减少阶段等待，以及为什么收益不会无限增加。实践必须保留独立输入、运行结果和结论。

In [ ]:
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))
from scheduler import pipeline_schedule, priority_schedule

payload = json.loads((data_dir / 'input.json').read_text(encoding='utf-8'))
priority_order = priority_schedule(payload['parent'], payload['cost'])
measurements = {}
for depth in (1, 2, 3):
    timing = pipeline_schedule(
        priority_order,
        payload['cost'],
        queue_depth=depth,
        copy_in=payload['copy_in'],
        copy_out=payload['copy_out'],
        compute_lanes=payload['compute_lanes'],
    )
    measurements[depth] = timing['end_to_end']
    print(f'queue_depth={depth}: end_to_end={timing["end_to_end"]:.1f}')

assert measurements[1] >= measurements[2] >= measurements[3]
print('结论：缓冲深度增加可以减少等待，但吞吐仍受 CopyIn、Compute 和 CopyOut 最慢阶段限制。')

完成实践后运行下一个代码单元查看参考答案。

In [ ]:
!cat answer/10.02_tree_queue_lab.md